In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib


In [3]:
paysim_path = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/data/paysim.csv"

df = pd.read_csv(paysim_path)
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [9]:
possible_columns = df.columns.tolist()
print(possible_columns)


['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [5]:
df.info()
print("\nFraud label distribution:")
print(df['isFraud'].value_counts())
print("\nFraud ratio:", df['isFraud'].mean())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB

Fraud label distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud ratio: 0.001290820448180152


In [11]:
features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",   # <-- corrected
    "oldbalanceDest",
    "newbalanceDest",
]

target = "isFraud"


In [15]:
df_model = df[features + [target]].copy()
df_model.head()



,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [17]:
df_sample = df_model.sample(n=200000, random_state=42)
df_sample['isFraud'].value_counts(), df_sample.shape


(isFraud
 0    199732
 1       268
 Name: count, dtype: int64,
 (200000, 7))

In [23]:
data = df_sample 


In [25]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep fraud ratio consistent
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()


((160000, 6), (40000, 6), 0.0013375, 0.00135)

In [27]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep fraud ratio consistent
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()


((160000, 6), (40000, 6), 0.0013375, 0.00135)

In [33]:
cat_features = ["type"]
num_features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",   # <-- corrected
    "oldbalanceDest",
    "newbalanceDest",
]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Features and target
X = data[features].copy()
y = data[target].astype(int)

# Categorical vs numeric
cat_features = ["type"]
num_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

# ColumnTransformer: ONE-HOT for 'type', SCALER for numeric columns
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ],
    remainder="drop"   # do not pass through anything else
)

log_reg = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        max_iter=200,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit model
log_reg.fit(X_train, y_train)

# Evaluate
y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     39946
           1       0.02      0.98      0.04        54

    accuracy                           0.94     40000
   macro avg       0.51      0.96      0.50     40000
weighted avg       1.00      0.94      0.97     40000

ROC-AUC: 0.9876221788303099
